# 7 — The temporal central charge of the Alcaraz model

Notebook 6 validated the transverse framework end-to-end on the integrable Ising chain: $c=1/2$
recovered from the entropy chord, the $\pi/24$ constant, and the $\lambda_0$ circle. We now apply
the *same* program to the genuinely non-integrable case, $p=0.1$, and ask the central question of
the thesis: **does the temporal central charge stay at $1/2$ when integrability is broken?**

An earlier attempt reported this as a failure ($c\approx0.69\to6.0$). Notebook 3 showed *why*: the
single-vector power method — and an **under-converged block power method (`itermax=800`)** — stall
at the transfer-matrix gap closing, contaminating the eigenvectors and the entropy profile built
from them. The **well-converged block PM (`itermax=8000`)** stays clean through the gap closing,
reaches higher $T$, and restores the conformal dome. This notebook redoes the extraction with that
method.

## The program: two routes, with $p=0$ as calibration

For $p>0$ the ANNNI MPO is **asymmetric** (the upper-triangular FSM of notebook 2), so
$|L\rangle\neq|R^*\rangle$ and the symmetric $n\to1$ machinery of notebook 6 is unavailable. We use
the **block power method** (left/right eigenvectors) and read the central charge **two ways**:

1. **Rényi-2 entropy profile** — $S_2^{\rm gen}(t)$ from `gen_renyi2(L,R)`, whose CFT coefficient is
   $c/8$ (i.e. $\tfrac{c}{12}\tfrac{n+1}{n}$ at $n=2$), so $c = 8\times\text{slope}$. Uses the full
   eigenvectors.
2. **Leading transfer-matrix eigenvalues** — the more robust route (eigenvalues survive the gap
   closing better than eigenvectors). The paper's Eq. (3),
   $\mathrm{Im}(\lambda_0)=a v T-\kappa/(vT)$ with $\kappa=-\pi c\,\delta t/6$, gives $c$ from the
   $1/T^2$ correction; Eq. (4), $\mathrm{Im}(\lambda_1-\lambda_0)=-\pi x_1/(vT)$, gives the boundary
   exponent; and the dominant eigenvalue traces a constant-radius **circle** (emergent dual
   unitarity).

Crucially, the $p=0$ run goes through the **identical asymmetric pipeline** as a calibration: if
the method is clean it must return $c=1/2$ there. The convention-free headline is therefore the
**$p{=}0.1/p{=}0$ ratio** — independent of the velocity normalization and the $\delta t$ prefactor.

Both routes are fed by **one** master block-PM sweep with $\beta_0=0.2$ (`nbeta=4`, the UV
regulator Eq. 3 requires).

In [1]:
include("src/thesislib.jl")
using LsqFit, Printf

dt = 0.1
v  = 2.0    # Ising sound velocity (paper: v=2). Assumed for p>0 too — see the velocity caveat below.

# Phase unwrapping for Im(λ) sequences vs T.
function unwrap(vv)
    out = copy(vv)
    for i in 2:length(vv)
        while out[i]-out[i-1] >  pi; out[i] -= 2pi; end
        while out[i]-out[i-1] < -pi; out[i] += 2pi; end
    end
    out
end

# Leading-eigenvalue phase, rotated by π to keep it away from the ±π branch cut
# (the Alcaraz transfer eigenvalues sit near the negative real axis — nb5).
ph(t) = angle(-t)

ph (generic function with 1 method)

## Lightweight eigenvalue sweep (Route 2 — fast)

Route 2 (circle, Eq. 3, Eq. 4) depends only on the **eigenvalues** $\lambda_0,\lambda_1$, not on the
eigenvectors. Eigenvalues converge much faster than eigenvectors — so a cheap single-vector
`powermethod_lr` already gives a reliable $\lambda_0$ even through the gap closing, and a quick
`block_transfer_eigs(k=2, itermax=400)` pins $\lambda_1$.

This lightweight sweep runs first to give Route 2 results immediately. If the heavy block-PM sweep
below has been run, its eigenvalues take precedence (the `done` dict merges both caches with the
heavy one overriding). Route 1 (entropy profiles) still requires the heavy sweep because it needs
clean eigenvectors.

In [ ]:
# Lightweight Route 2 sweep — eigenvalues only via block_transfer_eigs(k=2, itermax=400) (CHEAP).
# Gives λ₀,λ₁ reliably even through the gap closing (eigenvalues converge before eigenvectors).
# The expensive block-PM sweep below supersedes this once run, but this gets Route 2 results fast.
p_list    = [0.0, 0.1]
T_ladder  = [2.0, 3.0, 4.0, 5.0, 6.0, 8.0, 10.0]
nbeta     = 4
cachefile_lite = "results/data/nb7_alcaraz_lite.jld2"

done_lite = isfile(cachefile_lite) ? load(cachefile_lite, "done") : Dict{Tuple{Float64,Float64},Any}()
for p in p_list, T in T_ladder
    haskey(done_lite, (p,T)) && continue
    try
        mpo, scaffold = build_alcaraz_tmpo(T; p=p, lambda=1.0, dt=dt, nbeta=nbeta, MPO_alg="VD2")
        theta, _, _, info = block_transfer_eigs(mpo, scaffold; k=2, maxdim=256, cutoff=1e-12,
                                                itermax=400, eps_conv=1e-6, n_track=2)
        done_lite[(p,T)] = (p=p, T=T, theta=collect(theta), reason=string(info[:reason]))
        @info "LITE  p=$p T=$T  |θ₀|=$(round(abs(theta[1]),digits=5))  reason=$(info[:reason])"
    catch err
        @warn "LITE  p=$p T=$T failed: $err"; done_lite[(p,T)] = (error=string(err),)
    end
    jldsave(cachefile_lite; done=done_lite); GC.gc()
end

# Make the lite results available under the same done/ok/Ts_ok interface used by Route 2 cells.
# If the heavy block-PM cache exists, prefer it; otherwise fall back to lite.
cachefile_heavy = "results/data/nb7_alcaraz_block.jld2"
done_heavy = isfile(cachefile_heavy) ? load(cachefile_heavy, "done") : Dict{Tuple{Float64,Float64},Any}()
done = merge(done_lite, done_heavy)    # heavy overrides lite where both exist

ok(p,T)  = haskey(done,(p,T)) && !haskey(done[(p,T)], :error)
Ts_ok(p) = sort([T for (pp,T) in keys(done) if pp==p && ok(p,T)])

src = isempty(done_heavy) ? "lite (block k=2, iter=400)" : "heavy block PM preferred"
@info "Route 2 data source: $src"
@printf("%-6s %-6s %-10s %-10s %-12s %s\n","p","T","|θ₀|","|θ₁|","gapratio","reason")
for p in p_list, T in Ts_ok(p)
    th = done[(p,T)].theta
    @printf("%-6.1f %-6.1f %-10.5f %-10.5f %-12.5f %s\n",
            p, T, abs(th[1]), abs(th[2]), abs(th[2])/max(abs(th[1]),1e-30), done[(p,T)].reason)
end

[ Info: Checking symmetry MPO tensor on physical(space) => bond(time) indices
[ Info: Tensor symmetric (dim=2|id=926|"S=1/2,Site") <-> (dim=2|id=926|"S=1/2,Site")'
[ Info: Checking symmetry MPO tensor on bond(space) => phys(time) indices
┌ Warning: Tensor *not* symmetric (dim=3|id=537|"Link,l=1") <-> (dim=3|id=487|"Link,l=2"), normdiff = 0.944688073151956
└ @ ITransverse ~/.julia/packages/ITransverse/8pmYI/src/ITenUtils/itensor_utils.jl:93
[ Info: Checking symmetry MPO tensor on physical(space) => bond(time) indices
[ Info: Tensor symmetric (dim=2|id=902|"S=1/2,Site") <-> (dim=2|id=902|"S=1/2,Site")'
[ Info: Checking symmetry MPO tensor on bond(space) => phys(time) indices
┌ Warning: Tensor *not* symmetric (dim=3|id=21|"Link,l=1") <-> (dim=3|id=431|"Link,l=2"), normdiff = 0.10493384159131464
└ @ ITransverse ~/.julia/packages/ITransverse/8pmYI/src/ITenUtils/itensor_utils.jl:93
[ Info: LITE  p=0.0 T=2.0  |θ₀|=1.49635  reason=converged
[ Info: Checking symmetry MPO tensor on physical(spac

## Heavy block-PM sweep (Route 1 — entropy profiles + supersedes lite eigenvalues)

Route 1 needs the full eigenvectors $(\langle L|,|R\rangle)$ from a **well-converged** block PM
(`itermax=8000`). This cell is expensive; skip it on the first pass and come back once Route 2 looks
healthy. When it runs, its eigenvalues also supersede the lite sweep above (the `done` dict merges
both, heavy wins).

In [ ]:
# Master sweep — leading eigenvalues + Rényi-2 profile per (p,T). HEAVY; crash-safe per point.
# Skip this cell on first pass (Route 2 works from the lite sweep). Run later for Route 1 profiles.
p_list    = [0.0, 0.1]
T_ladder  = [2.0, 3.0, 4.0, 5.0, 6.0, 8.0, 10.0]   # extend to 12.0, 14.0 if trends are clean
nbeta     = 4                                       # β0 = 0.2 (mandatory for Eq.3)
cachefile = "results/data/nb7_alcaraz_block.jld2"

done_block = isfile(cachefile) ? load(cachefile, "done") : Dict{Tuple{Float64,Float64},Any}()
for p in p_list, T in T_ladder
    haskey(done_block, (p,T)) && continue
    try
        mpo, scaffold = build_alcaraz_tmpo(T; p=p, lambda=1.0, dt=dt, nbeta=nbeta, MPO_alg="VD2")
        theta, L, R, info = block_transfer_eigs(mpo, scaffold; k=4, maxdim=256, cutoff=1e-12,
                                                itermax=8000, eps_conv=1e-6, n_track=2)
        s2   = ITransverse.gen_renyi2(L[1], R[1]); half = nbeta ÷ 2
        done_block[(p,T)] = (p=p, T=T, theta=collect(theta),
                       s2=collect(s2[half+1:end-half]), reason=string(info[:reason]))
        @info "p=$p T=$T done (reason=$(info[:reason]))"
    catch err
        @warn "p=$p T=$T failed: $err"; done_block[(p,T)] = (error=string(err),)
    end
    jldsave(cachefile; done=done_block); GC.gc()
end

# Merge: heavy overrides lite, then refresh the shared done/ok/Ts_ok.
done = merge(done_lite, done_block)
ok(p,T)  = haskey(done,(p,T)) && !haskey(done[(p,T)], :error)
Ts_ok(p) = sort([T for (pp,T) in keys(done) if pp==p && ok(p,T)])

@printf("%-6s %-6s %-10s %-10s %-12s %s\n","p","T","|θ₀|","|θ₁|","gapratio","reason")
for p in p_list, T in Ts_ok(p)
    th = done[(p,T)].theta
    @printf("%-6.1f %-6.1f %-10.5f %-10.5f %-12.5f %s\n",
            p, T, abs(th[1]), abs(th[2]), abs(th[2])/max(abs(th[1]),1e-30), done[(p,T)].reason)
end

## Route 1 — the Rényi-2 entropy profiles

With the well-converged block PM the $S_2^{\rm gen}(t)$ domes should hold the conformal **chord**
shape $\propto\log[(2T/\pi)\sin(\pi t/T)]$ up to larger $T$ than the single-vector method allowed in
notebook 3. We overlay the profiles for $p=0$ (calibration) and $p=0.1$ side by side.

In [ ]:
# Block-PM Rényi-2 domes for several T (both p).
function plot_domes(p)
    Ts = Ts_ok(p); cols = cgrad(:viridis, max(length(Ts),2), categorical=true)
    plt = plot(title="Re S₂ profiles — Alcaraz p=$p (block PM)", xlabel="t/T", ylabel="Re S₂",
               framestyle=:box, grid=true, legend=:outerright)
    for (i,T) in enumerate(Ts)
        s2 = real.(done[(p,T)].s2); x = (1:length(s2)) ./ (length(s2)+1)
        plot!(plt, x, s2; label="T=$T", lw=2, color=cols[i])
    end
    plt
end
plot(plot_domes(0.0), plot_domes(0.1); layout=(1,2), size=(1300,460), margin=5Plots.mm)

### Rényi-2 central charge $c = 8\times\text{slope}$

We fit each dome's central half to the chord and read $c=8\times\text{slope}$. The $p=0$ column is
the calibration: a clean asymmetric pipeline returns $c(p{=}0)\approx1/2$. The convention-free
headline is the **calibrated** $c(p{=}0.1)=0.5\,\times\,\text{slope}(p{=}0.1)/\text{slope}(p{=}0)$.

In [ ]:
# c = 8·slope (fit central half of the dome). Calibrate p=0 → 0.5, report the ratio.
function renyi2_slope(s2_re, T)
    Nb = length(s2_re); x = (1:Nb) ./ (Nb+1)
    xc = log.((2T/pi) .* sin.(pi .* x))
    lo = round(Int,0.25Nb)+1; hi = round(Int,0.75Nb)
    @. lin(xx,q) = q[1]*xx + q[2]
    curve_fit(lin, xc[lo:hi], s2_re[lo:hi], [1/16, 0.0]).param[1]
end
slope(p,T) = renyi2_slope(real.(done[(p,T)].s2), T)

@printf("Rényi-2 c = 8·slope per T (calibration c(p=0) should be ≈ 0.5 if the pipeline is clean):\n")
@printf("%-6s  %-14s  %-14s\n","T","c(p=0)","c(p=0.1)")
for T in T_ladder
    c0 = ok(0.0,T) ? @sprintf("%.4f", 8*slope(0.0,T)) : "—"
    c1 = ok(0.1,T) ? @sprintf("%.4f", 8*slope(0.1,T)) : "—"
    @printf("%-6.1f  %-14s  %-14s\n", T, c0, c1)
end

Tsc  = [T for T in T_ladder if ok(0.0,T) && ok(0.1,T)]
c0v  = [8*slope(0.0,T) for T in Tsc]
c1v  = [8*slope(0.1,T) for T in Tsc]
ccal = [0.5*slope(0.1,T)/slope(0.0,T) for T in Tsc]   # calibrated c(p=0.1)
plt = plot(Tsc, c0v; label="c(p=0) raw", marker=:circle, lw=2, color=:blue, xlabel="T",
           ylabel="c (Rényi-2, = 8·slope)", framestyle=:box, grid=true,
           title="Temporal central charge vs T", legend=:topright)
plot!(plt, Tsc, c1v;  label="c(p=0.1) raw", marker=:square, lw=2, color=:red)
plot!(plt, Tsc, ccal; label="c(p=0.1) calibrated (0.5·slope-ratio)", marker=:diamond, lw=2, color=:green)
hline!(plt, [0.5]; ls=:dash, color=:gray, label="c = 1/2")
plt

## Route 2 — the leading eigenvalues

### Emergent dual unitarity: the $\lambda_0$ circle

The paper notes that the dominant eigenvalue distributes "on an almost constant-radius circle" as
$T$ grows — the transfer matrix tending to a rescaled unitary. We plot $\Lambda_0(T)$ in the complex
plane for $p=0$ and $p=0.1$ against the unit circle, mirroring the notebook-6 Ising circle.

In [ ]:
# Leading eigenvalue Λ₀(T) in the complex plane (both p) — near-constant radius ⇒ dual unitarity.
phi = range(0, 2pi, length=400)
plt = plot(cos.(phi), sin.(phi); ls=:dash, color=:gray, label="unit circle", aspect_ratio=:equal,
           xlabel="Re Λ₀", ylabel="Im Λ₀", framestyle=:box, grid=true, legend=:outerright,
           title="Leading eigenvalue Λ₀(T): p=0 vs p=0.1")
for (p,col,mk) in zip([0.0,0.1],[:blue,:red],[:circle,:square])
    L0 = [done[(p,T)].theta[1] for T in Ts_ok(p)]
    rbar = isempty(L0) ? NaN : sum(abs,L0)/length(L0)
    scatter!(plt, real.(L0), imag.(L0); marker=mk, ms=6, color=col,
             label="p=$p  (|Λ₀|≈$(round(rbar,digits=3)))")
end
plt

### Eq. (4) — the boundary exponent $x_1$

$\mathrm{Im}(\lambda_1-\lambda_0)=-\pi x_1/(vT)$ depends only on a *phase difference*, so the
non-universal constants cancel and the fit is robust. For our free boundary $|X+\rangle$ the paper
predicts $x_1=1/2$ (with $a_1\approx\pi/4$). This is the most reliable probe and a consistency check
against the confirmed result.

In [ ]:
# Eq.(4): Im(λ1−λ0) = −π x1/(vT).  Fit a1/T + a3/T³ ; x1 = |v·a1/π|.
function eq4(p; v=2.0)
    Ts  = Ts_ok(p)
    th0 = [done[(p,T)].theta[1] for T in Ts]; th1 = [done[(p,T)].theta[2] for T in Ts]
    g   = [mod(ph(t1)-ph(t0)+pi, 2pi)-pi for (t0,t1) in zip(th0,th1)]   # wrapped phase gap
    @. mg(T,q) = q[1]/T + q[2]/T^3
    fg = curve_fit(mg, Float64.(Ts), g, [-1.0, 0.0])
    (; Ts, g, fit=fg, a1=fg.param[1], x1=abs(v*fg.param[1]/pi))
end
e0 = eq4(0.0; v=v); e1 = eq4(0.1; v=v)
println("Eq.(4) boundary exponent (free-BC |X+⟩ ⇒ expect x1 ≈ 1/2):")
@printf("  p=0    : x1 = %.3f  (a1=%.3f; paper free-BC a1≈π/4=%.3f)\n", e0.x1, e0.a1, pi/4)
@printf("  p=0.1  : x1 = %.3f  (a1=%.3f)\n", e1.x1, e1.a1)

plt = plot(title="Eq.(4) first gap Im(λ₁−λ₀)", xlabel="T", ylabel="Im(λ₁−λ₀)",
           framestyle=:box, grid=true, legend=:bottomright)
scatter!(plt, e0.Ts, e0.g; label="p=0", color=:blue, ms=5)
scatter!(plt, e1.Ts, e1.g; label="p=0.1", color=:red, ms=5)
mg4(T,q) = q[1]./T .+ q[2]./T.^3; Td = range(minimum(e0.Ts), maximum(e0.Ts), length=200)
plot!(plt, Td, mg4(Td,e0.fit.param); label="p=0 fit (x1=$(round(e0.x1,digits=2)))", color=:blue, lw=2)
plot!(plt, Td, mg4(Td,e1.fit.param); label="p=0.1 fit (x1=$(round(e1.x1,digits=2)))", color=:red, lw=2)
plt

### Eq. (3) — the central charge from $\lambda_0$

From $\mathrm{Im}(\lambda_0)=a v T-\kappa/(vT)$ with $\kappa=-\pi c\,\delta t/6$, dividing by $T$
gives $\mathrm{Im}(\lambda_0)/T = a_0 + a_2/T^2$ with $a_2=-\kappa/v$, hence
$c = 6\,v\,|a_2|/(\pi\,\delta t)$. The absolute prefactor depends on the $\delta t$/per-step
convention, so the **primary** report is the calibration: fix $c(p{=}0)=1/2$ to pin the prefactor
$K$ ($c=K|a_2|$), then $c(p{=}0.1)=0.5\,|a_2(p{=}0.1)|/|a_2(p{=}0)|$. The nominal
$6v|a_2|/(\pi\delta t)$ is shown alongside as a cross-check (its value on the $p=0$ run also tells
us whether $v=2$ and the prefactor convention are right).

In [ ]:
# Eq.(3): fit Im(λ0)/T = a0 + a1/T + a2/T² ; κ=−v·a2 ; c_nominal = 6v|a2|/(πδt). Calibrate on p=0.
function eq3_a2(p)
    Ts   = Ts_ok(p)
    lam0 = [done[(p,T)].theta[1] for T in Ts]
    y    = unwrap([ph(t) for t in lam0]) ./ Ts
    @. m3(T,q) = q[1] + q[2]/T + q[3]/T^2
    f = curve_fit(m3, Float64.(Ts), y, [1.0, 0.0, 0.01])
    (; Ts, y, fit=f, a2=f.param[3])
end
q0 = eq3_a2(0.0); q1 = eq3_a2(0.1)
c_nominal(a2) = 6*v*abs(a2)/(pi*dt)
println("Eq.(3) central charge from the leading eigenvalue:")
@printf("  p=0    : a2 = %+.5f   c_nominal = %.3f   (= 6v|a2|/(πδt), v=%.1f)\n", q0.a2, c_nominal(q0.a2), v)
@printf("  p=0.1  : a2 = %+.5f   c_nominal = %.3f\n", q1.a2, c_nominal(q1.a2))
@printf("  CALIBRATED (p=0 ≡ 1/2):  c(p=0.1) = 0.5·|a2(0.1)|/|a2(0)| = %.3f\n",
        0.5*abs(q1.a2)/abs(q0.a2))

plt = plot(title="Eq.(3) Im(λ₀)/T fit", xlabel="T", ylabel="Im(λ₀)/T",
           framestyle=:box, grid=true, legend=:topright)
scatter!(plt, q0.Ts, q0.y; label="p=0", color=:blue, ms=5)
scatter!(plt, q1.Ts, q1.y; label="p=0.1", color=:red, ms=5)
m3p(T,q) = q[1] .+ q[2]./T .+ q[3]./T.^2; Td = range(minimum(q0.Ts), maximum(q0.Ts), length=200)
plot!(plt, Td, m3p(Td,q0.fit.param); label="p=0 fit", color=:blue, lw=2)
plot!(plt, Td, m3p(Td,q1.fit.param); label="p=0.1 fit", color=:red, lw=2)
plt

## Verdict

Read the headline off the **calibrated** numbers, which cancel the velocity and prefactor
conventions by tying $p=0$ to the known $c=1/2$:

- **Calibration sanity.** First check that the $p=0$ column is healthy: the Rényi-2 $c(p{=}0)$ and
  the Eq. (3) $c_{\rm nominal}(p{=}0)$ should both sit near $1/2$, and the $p=0$ entropy domes should
  follow the chord. If they do, the asymmetric block-PM pipeline is now clean (the under-convergence
  of the old attempt is fixed by `itermax=8000`); if not, the residual is honest method error and
  the $p>0$ numbers inherit it.
- **Headline — does universality survive?** The convention-free quantities are the
  $p{=}0.1/p{=}0$ slope ratio (Route 1) and $|a_2|$ ratio (Route 2). A ratio $\approx1$ — i.e.
  calibrated $c(p{=}0.1)\approx1/2$ — together with $x_1\approx1/2$ (Eq. 4, already robust) and a
  constant-radius $\lambda_0$ circle would say the temporal Ising universality **survives the NNN
  frustration**. A clear departure would locate the breakdown.

**Velocity caveat.** Both the absolute Eq. (3)/(4) numbers assume $v=2$ (Ising). For $p>0$ the sound
velocity may drift; the calibrated ratios are insensitive to an overall $v$ only if $v$ is the same
at both $p$. The Eq. (4) result $x_1(p{=}0.1)\approx1/2$ is itself evidence that $v\approx2$ still
holds at $p=0.1$. Extending `T_ladder` (12, 14) and a $\delta t$-convergence check are the natural
next refinements once these trends are established.